<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/cracked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cracked tool router with LlamaIndex

Give a LlamaIndex agent 60,000+ tools through one API key.

## Overview

[Cracked](https://cracked.ai) is a tool router for agents. One API key and one prepaid balance give an agent web search and scraping, Instagram, TikTok, YouTube, X, Reddit and LinkedIn data, Google Maps places and reviews, Amazon products, company enrichment, jobs, real estate, stock and crypto quotes, weather, and AI models for text, image, video and speech. Every run is billed per call; there are no provider signups.

The workflow is **discover** a tool in natural language, **inspect** its JSON Schema, **run** it, and **poll** when the run is asynchronous. When the task matches a capability id (`web-search`, `instagram-profile`, `weather-forecast`, ...), a **smart run** does the routing in one call and falls back to the next provider on failure.

This notebook wraps the zero-dependency [`cracked-ai`](https://pypi.org/project/cracked-ai/) client as LlamaIndex `FunctionTool`s and runs them in a `FunctionAgent`.

## Setup

Install the Cracked client, LlamaIndex core and an LLM integration:

In [ ]:
%pip install cracked-ai llama-index-core llama-index-llms-openai

Get a Cracked key at [cracked.ai/app/keys](https://cracked.ai/app/keys), or let the agent register its own workspace (trial credit included) with `Cracked.register_agent(name="llamaindex-demo")`. The client reads `CRACKED_API_KEY` from the environment.

In [ ]:
import getpass
import os

if "CRACKED_API_KEY" not in os.environ:
    os.environ["CRACKED_API_KEY"] = getpass.getpass("Cracked API key: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

## Build the tools

Four functions cover the whole workflow. `input_json` is a JSON object string so the tool schema stays simple for every LLM; the functions return compact JSON strings and truncate large scraper output.

In [ ]:
import json

from cracked_ai import Cracked, CrackedError
from llama_index.core.tools import FunctionTool

client = Cracked()  # reads CRACKED_API_KEY; optional CRACKED_BASE_URL
MAX_CHARS = 20_000


def _safe(fn):
    try:
        return fn()
    except CrackedError as e:
        # 402 = top up at https://cracked.ai/app/billing; 503 BLOCKED = provider needs your own key, pick another
        return json.dumps(
            {"error": e.code, "status": e.status, "message": e.message}
        )


def cracked_discover(query: str, limit: int = 5) -> str:
    """Find tools on Cracked in natural language. Returns provider, endpoint, price, status and whether the tool is async."""
    keys = (
        "provider",
        "endpoint",
        "name",
        "price",
        "status",
        "verified",
        "async",
    )
    return _safe(
        lambda: json.dumps(
            [
                {k: h.get(k) for k in keys}
                for h in client.discover(query, limit=limit)["results"]
            ]
        )
    )


def cracked_inspect(provider: str, endpoint: str) -> str:
    """Read the JSON Schema of a tool's input before running it. Never guess field names."""
    return _safe(
        lambda: json.dumps(client.inspect(provider, endpoint)["input"]["body"])
    )


def cracked_run(provider: str, endpoint: str, input_json: str) -> str:
    """Run one provider endpoint. input_json is a JSON object string shaped by cracked_inspect. Returns status, cost and output."""

    def go():
        rec = client.run(provider, endpoint, json.loads(input_json))
        return json.dumps(
            {
                "status": rec["status"],
                "costUsd": rec["billing"]["totalUsd"],
                "output": rec["output"],
            }
        )[:MAX_CHARS]

    return _safe(go)


def cracked_run_capability(capability: str, input_json: str) -> str:
    """Smart run: Cracked picks the best live provider for a capability id (web-search, web-scrape, news-search, instagram-profile, google-maps-places, weather-forecast, stock-quote, llm-chat, ...) and falls back on failure. input_json is a JSON object string."""

    def go():
        rec = client.run_capability(capability, json.loads(input_json))
        return json.dumps(
            {
                "status": rec["status"],
                "routedTo": rec.get("routedTo"),
                "costUsd": rec["billing"]["totalUsd"],
                "output": rec["output"],
            }
        )[:MAX_CHARS]

    return _safe(go)


tools = [
    FunctionTool.from_defaults(fn)
    for fn in (
        cracked_discover,
        cracked_inspect,
        cracked_run,
        cracked_run_capability,
    )
]
[t.metadata.name for t in tools]

['cracked_discover',
 'cracked_inspect',
 'cracked_run',
 'cracked_run_capability']

## Run an agent

The agent discovers a weather tool, reads its schema, runs it and answers from the output.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

agent = FunctionAgent(
    tools=tools,
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt=(
        "Answer with live data. Find the right tool with cracked_discover, read its schema with "
        "cracked_inspect, then cracked_run it with the exact fields from the schema. When the task "
        "matches a capability id, cracked_run_capability is faster. Quote output fields; never invent data."
    ),
)

response = await agent.run("What is the weather in Austin, TX right now?")
print(response)

The current weather in Austin, TX is as follows:

- **Temperature:** 100.2°F
- **Apparent Temperature:** 105.3°F
- **Weather Description:** Mainly clear
- **Humidity:** 33%
- **Wind Speed:** 3.3 km/h
- **Wind Gusts:** 6.5 km/h
- **Cloud Cover:** 26%
- **Pressure:** 1009.7 hPa

The forecast for today includes slight rain showers with a maximum temperature of 100.2°F and a minimum of 77.9°F.


## Call a capability directly

Smart runs need no discover or inspect step. The public list of capability ids and their input fields is at https://cracked.ai/v1/capabilities.

In [ ]:
rec = client.run_capability(
    "web-search", {"query": "LlamaIndex FunctionAgent", "limit": 3}
)
print(rec["status"], rec["routedTo"], rec["billing"]["totalUsd"])
for r in rec["output"].get("results", [])[:3]:
    print("-", r.get("title"), r.get("url"))

COMPLETED {'provider': 'tavily', 'endpoint': '/search'} 0.011
- Agents | Developer Documentation https://developers.llamaindex.ai/python/framework/understanding/putting_it_all_together/agents
- Agent Classes - LlamaIndex https://developers.llamaindex.ai/python/framework-api-reference/agent
- FunctionAgent / AgentWorkflow Basic Introduction | Developer Documentation https://developers.llamaindex.ai/python/examples/agent/agent_workflow_basic


## Async runs and rules

- Scrapers marked `async: true` in discover results can take minutes. Submit with `client.run(provider, endpoint, input, wait=False)` (HTTP 202, status `RUNNING`) and finish with `client.wait_for(run_id, timeout=600)` or `client.get_run(run_id, wait=30)` in a loop; `client.stop(run_id)` aborts.
- Terminal statuses: `COMPLETED`, `FAILED`, `BLOCKED`, `STOPPED`, `TIMED_OUT`. `COMPLETED` with `providerResponse.httpStatus == 404` means the provider found nothing; it is not an error.
- `BLOCKED` (HTTP 503, not billed) means the provider needs your own key: add it at https://cracked.ai/app/connections or pick another result.
- Per-result tools bill per row returned, so set `limit` deliberately. `client.balance()` shows the remaining balance.

More: [Cracked docs](https://cracked.ai/docs), [`cracked-ai` on PyPI](https://pypi.org/project/cracked-ai/), [OpenAPI spec](https://cracked.ai/openapi.json).